# Getting the laws' discussion of the Deputies' Chamber

The Deputies' Chamber has the role of discussing laws that are proposed to them by different paths (mainly from Deputies' themselves, the Senate or by Presidential decrees). For each discussion of each law, the National Congress' Library ("*BCN*" from the Spanish name) saves the transcript of the discussion, storing in it the name and speech of each intervention.
An image on how it is stored will be added in the written report.

The goal of this notebook is to extract all of the relevant discussion of the law we provided. We had to add a limit on the date of when these discussion took place so the majority of speakers where the deputies we scrapped information of

In [ ]:
# import chunk

import re # to use regex
import json # used to save the scraped data in json format, which I still need to prove useful for this project
import time # used to add pauses (delays) between request to the server
import random # used to add randomness to the request, making it look more like a human and not like a bot
from pathlib import Path # used to work with file paths, important to save the files in the right place
from datetime import datetime, date # used to work with date times
import xml.etree.ElementTree as ET

import requests
import pandas as pd
from bs4 import BeautifulSoup

Configuration previous to the webscrapping

In [ ]:
# laws to be scraped
LAW_URLS = [
    "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8431/",
    "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8429/",
    "https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8425/"
]

# date range that we consider
START_DATE = date(2022, 3, 11) # YYYY, MM, DD
END_DATE   = date(2026, 3, 10)

# outputs where to save the results
BASE_DIR = Path("../../data/extracted_laws") # folders
BASE_DIR.mkdir(parents=True, exist_ok=True)


SUMMARY_CSV = Path("../../data/discussion_summary.csv")

# headers for the HTTP requests
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)


SKIP_IF_ALREADY_SAVED = False
PAUSE_BETWEEN_POSTS = (8.0, 15.0)   # range for random pause between transcript requests
PAUSE_BETWEEN_LAWS = (20.0, 40.0)   # range for random pause between laws
SAVE_RAW_HTML = True # optional



## Functions
Since several steps here are repeated, I will create some helper functions to avoid code repetition and make the code cleaner. These functions will be used in the main scraping loop, but they can also be useful for debugging and testing individual steps.

In [ ]:
# helper functions

# eliminates whitespace so the text is cleaners
def normalize_space(text):
    """Collapse repeated whitespace and replace non-breaking spaces."""
    if text is None: # this is here for robustness, just to avoid errors
        return ""
    text = text.replace("\xa0", " ") # a non-breaking space is a special character that looks like a space but prevents line breaks, we replace it with a normal space
    return " ".join(text.split())

# this function is fundamental to avoid getting blocked by the server by adding some randomess to the requests
def pause_random(seconds_range):
    """Sleep for a random number of seconds inside the given range tuple."""
    low, high = seconds_range
    delay = random.uniform(low, high)
    # we print the delay so we can see how long we're waiting
    print(f"Sleeping {delay:.1f}s...")
    time.sleep(delay)

In [ ]:
# functions to get raw HTML, soup and metadata

def get_law_page_metadata(session, law_url, timeout=30):
    """
    Download the law page (once) and return the parsed HTML soup object, the raw HTML and the basic law metadata (BCN id, number, title).
    """
    response = session.get(law_url, timeout=timeout)   
    
    # This print allow to see when (if so) an 405 status code appears
    print(f"GET {law_url} status code: {response.status_code}")
    
    response.raise_for_status()
    
    # store the whole HTML (helpful for debugging)
    html = response.text
    # parse the HTML 
    soup = BeautifulSoup(html, "html.parser")

    # Get the metadata
    page_text = soup.get_text("\n", strip=True)

    # law number
    law_number_matched = re.search(r"Historia de la Ley N°\s*([0-9.]+)", page_text)
    law_number = law_number_matched.group(1) if law_number_matched else None

    # for the title extraction we use the HTML node
    title_node = soup.select_one('#hleyCompleta p.texto-justificado[itemprop="description"]')
    law_title = normalize_space(title_node.get_text(" ", strip=True)) if title_node else None

    # Extracts the BCN law id from the url
    m = re.search(r"/historia-de-la-ley/(\d+)/?", law_url)
    law_id = m.group(1) if m else None

    law_meta = {
        "law_id": law_id,
        "law_number": law_number,
        "law_title": law_title,
        "history_url": law_url,
    }

    return soup, html, law_meta

In [ ]:
# functions to store the results in a structured way

def safe_law_number(law_number, fallback="unknown_law"):
    """Convert a law number like 21.755 into a folder-safe version like 21755."""
    if not law_number:
        return fallback
    return law_number.replace(".", "")

def get_law_folder(base_dir, law_meta):
    """
    Create one folder per law.
    """
    folder_name = f"law_{safe_law_number(law_meta.get('law_number'))}"
    law_folder = base_dir / folder_name
    law_folder.mkdir(parents=True, exist_ok=True)
    return law_folder

#### discover discussions

Each law can start and take several different paths depending of the place that it starts, if it get's rejected, accepted or is subjectd to indications. That's why we need to develop a function capable to check the table of content of the page to collect each Discussion ("Discusión en Sala")

In [ ]:
def discover_discussions_from_index(soup):
    """
    Parse the Índice tree (called "#arbol") and collect every 'Discusión en Sala' entry, also to which stage it belongs to. Returns basic information of the stage
    """
    arbol = soup.select_one("#arbol")
    
    # kept for security
    if arbol is None:
        print("Could not find #arbol (Índice) in the page HTML.")

    # the anchor are the nodes that contain the text of the stages and items
    anchor_texts = [
        normalize_space(a.get_text(" ", strip=True)) # we get the text of each anchor, normalize the spaces and store it in a list
        for a in arbol.select("a")
    ]

    # stay only with non-empty texts, because the page sometimes has empty anchors
    anchor_texts = [x for x in anchor_texts if x]

    # this regexs is used to identify the stages and the items in the Índice, we will use it to extract the relevant information 
    stage_re = re.compile(r"^(\d+)\.\s+(.*)$")
    item_re = re.compile(r"^(\d+)\.(\d+)\.\s+(.*?)\s*\((\d{2}-\d{2}-\d{4})\)$") 


    rows = [] # list to be filled with dictionaries
    current_stage_num = None
    current_stage_text = None

    # for each tect in the anchors we check if it is a stage or an item, and if it is a discussion in sala, we store the relevant information in a list of dictionaries
    for text in anchor_texts:

        # Work on the stage leve
        # Example: "1. Primer Trámite Constitucional: Cámara de Diputados"
        m_stage = stage_re.match(text)
        if m_stage and "Trámite" in text:
            current_stage_num = int(m_stage.group(1)) # in the example: 1
            current_stage_text = m_stage.group(2) # in the example: "Primer Trámite Constitucional: Cámara de Diputados"
            continue

        # work on item level
        # Example: "1.5. Discusión en Sala (05-03-2025)"
        m_item = item_re.match(text)
        if m_item and current_stage_num is not None:
            stage_num = int(m_item.group(1)) # in example: 1
            item_num = int(m_item.group(2)) # in example: 5
            label = m_item.group(3) # in example: "Discusión en Sala"
            date_str = m_item.group(4) # in example: "05-03-2025"

            # The item should belong to the current stage (because the composition of the section_key is stage_num-item_num)
            if stage_num != current_stage_num:
                continue

            if "discusión en sala" in label.lower():
                event_date = datetime.strptime(date_str, "%d-%m-%Y").date() # format: day-month-year
                stage_text_low = current_stage_text.lower()

                rows.append({
                    "stage_num": stage_num,
                    "stage_text": current_stage_text,
                    "stage_text_lower": stage_text_low, # necessary to identify the type of stage (constitutional, diputados, senado, etc.)
                    "section_key": f"{stage_num}-{item_num}", # important for later matching with the transcripts
                    "label": label,
                    "event_date": event_date, 
                    
                    # Check to avoid getting discussion stages that are not relevant for this iteration of the project
                    "stage_is_constitutional": "trámite constitucional" in stage_text_low,
                    "stage_has_diputados": "cámara de diputados" in stage_text_low,
                    "stage_has_senado": "senado" in stage_text_low,
                    "stage_is_mixed": ("diputados-senado" in stage_text_low or ("cámara de diputados" in stage_text_low and "senado" in stage_text_low)),
                    "stage_is_finalization": "finalización" in stage_text_low, # this stage is not relevant (has no discussion in sala)
                })

    return pd.DataFrame(rows)



# we use the previous function to filter and get the relevant discussion stages
def filter_target_discussions(df, start_date=START_DATE, end_date=END_DATE):
    """
    Filter to keep only discussions that:
    - are inside the targeted date period 
    - are Trámite Constitucional
    - are in Cámara de Diputados only
    - are not mixed Cámara+Senado
    - are not Trámite Finalización
    """
    if df.empty:
        return df.copy()

    # & is used instead of and because pandaseries use bitwise operators for elementwise logical operations
    mask = (
        (df["event_date"] >= start_date) & # later than our determined start date
        (df["event_date"] <= end_date) & # before our determined end date
        (df["stage_is_constitutional"]) & # are Trámite Constitucional
        (df["stage_has_diputados"]) & # are in Cámara de Diputados only
        (~df["stage_has_senado"]) & # are not in Senado
        (~df["stage_is_mixed"]) & # are not mixed Cámara+Senado
        (~df["stage_is_finalization"]) # are not Trámite Finalización
    )

    return df.loc[mask].copy()


#### Extracting the discussion

The BCN server, when clicked on "*Show full document*" ("*Ver documento completo*") to see the whole transcription of a discussion send a xajax request specifying what the user wants to see, as response the user get a response composed by 4 JavaScript commands, the second one contains the transcription that will be loaded into an overlay ('.modal-body' from Bootstrap). The goal of the following function is to obtain the transcription simulating the request that would be done from the browser from a user


In [ ]:
def extract_parse_modal_html(response_text):
    """
    The response is XML containing JS commands, we extract the JS command that injects the transcript HTML into $('.modal-body')
    """
    root = ET.fromstring(response_text) # parse the XML response
    cmd_nodes = root.findall(".//cmd[@n='js']") # finds all the nodes that are JS commands

    candidates = []
    for node in cmd_nodes:
        txt = node.text or ""  # get the text content of the node, or an empty string if it's None
        if "$('.modal-body').html(" in txt: # we look for the JS command that contains the injection of the HTML into $('.modal-body')
            candidates.append(txt)

    # in case we don't find any candidate
    if not candidates:
        print("No modal-body injection found in xajax response")

    # Prefer the longest candidate containing div.item, because it is where the transcript is located
    target_js = None
    for txt in sorted(candidates, key=len, reverse=True):
        if 'div class="item"' in txt: # this is how to find the candidate that contains the transcript
            target_js = txt
            break # break why we want the longest one with div.item, so once we find it we can stop

    if target_js is None: # if we still cant find any candidate with div.item, we take the longest one
        target_js = max(candidates, key=len)


    # Try single-quoted payload first
    start_token = "$('.modal-body').html('"
    end_token = "');"


    if start_token in target_js:
        start = target_js.find(start_token) + len(start_token) # really start where the HTML starts, not where the JS command starts
        end = target_js.rfind(end_token)
        raw_html = target_js[start:end]
    else:
        # Fallback for double quotes if needed
        start_token = "$('.modal-body').html(\""
        end_token = "\");"
        start = target_js.find(start_token)
        start += len(start_token)
        end = target_js.rfind(end_token)
        raw_html = target_js[start:end]

    # Unescape JS string escapes, if not the HTML will be hard to parse with BeautifulSoup
    modal_html = raw_html
    modal_html = modal_html.replace("\\\\", "\\")
    modal_html = modal_html.replace("\\'", "'")
    modal_html = modal_html.replace('\\"', '"')
    modal_html = modal_html.replace("\\n", "\n")
    modal_html = modal_html.replace("\\t", "\t")
    modal_html = modal_html.replace("\\r", "\r")

    
    # Once we obtain the modal_html with the previous function, we need to parse it to extract the transcript text and some metadata about it
    soup = BeautifulSoup(modal_html, "html.parser")

    item = soup.select_one("div.item") # this is how to find the node that contains the transcript
    container = item if item is not None else soup # if we find the div.item we use it as container, if not we use the whole soup as container

    # we extract the attibutes of the item
    attrs = dict(item.attrs) if item else {}

    paragraphs = [
        normalize_space(p.get_text(" ", strip=True))
        for p in container.find_all("p") # so we get the text of each paragraph, normalize the spaces and store it in a list
    ]
    # for security
    paragraphs = [p for p in paragraphs if p] 

    if paragraphs:
        transcript_text = "\n\n".join(paragraphs)
    else:
        transcript_text = normalize_space(container.get_text("\n", strip=True))


    # because some times there's a version that's not the official yet (it is still being worked on)
    upper_text = transcript_text.upper()
    preliminary_markers = ["LA VERSIÓN OFICIAL DE ESTA DISCUSIÓN EN SALA NO SE ENCUENTRA DISPONIBLE", "VERSIÓN PRELIMINAR"] 
    is_preliminary = any(marker in upper_text for marker in preliminary_markers)

    parsed = {
        "item_attrs": attrs,
        "paragraphs": paragraphs,
        "paragraph_count": len(paragraphs),
        "transcript_text": transcript_text,
        "char_count": len(transcript_text),
        "is_preliminary": is_preliminary,
    }

    return modal_html, parsed


# checking the network tab of the Web Developer Tools (in Firefox) I discovered how the page requests the transcript for each discussion in sala
def fetch_discussion_transcript(session, law_url, section_key, timeout=30):
    """
    Request a transcript (using the AJAX endpoint the page uses) for one section_key like '1-5' or '3-1'
    """
    
    # this is the AJAX request that the page does to get the transcript
    headers = {
        "User-Agent": USER_AGENT,
        "Referer": law_url,
        "Origin": "https://www.bcn.cl",
    }

    # this is the payload that the page sends in the POST request to get the transcript, 
    payload = [
        ("xajax", "verTextoCompleto"), # this mean "SeeFullText" in spanish
        ("xajaxr", str(int(time.time() * 1000))), # the page sends a timestamp in milliseconds, we replicate it to avoid suspicion
        ("xajaxargs[]", section_key), # this is the section we want to obtain
    ]

    # this is the POST request to the AJAX endpoint to get the transcript for the given section_key
    response = session.post(law_url, headers=headers, data=payload, timeout=timeout)

    response.raise_for_status()

    # we use the previous function to extract and parse the modal HTML
    modal_html, parsed = extract_parse_modal_html(response.text)


    return {
        "section_key": section_key,
        "modal_html": modal_html,
        **parsed, # ** means here means that we want to include all the key-value pairs from the parsed dictionary in the returned dictionary
    }

#### Saving the results

We decided to store all the transcripts in .txt but also in .json. txt is easier to read as a human, but .json allows to store more information and it's faster to work with

In [ ]:

def save_discussion_file(law_folder, law_meta, entry, transcript_payload):
    
    #Build output paths for one discussion.
    stamp = entry["event_date"].isoformat()
    key = entry["section_key"].replace("-", "_")

    txt_path = law_folder / f"discussion_{stamp}_{key}.txt"
    json_path = law_folder / f"discussion_{stamp}_{key}.json"



    # Save text in .txt file
    txt_path.write_text(transcript_payload["transcript_text"], encoding="utf-8")

    # Save metadata and transcript in a .JSON file
    bundle = {
        "law_meta": law_meta,
        "entry": {
            "stage_num": entry["stage_num"],
            "stage_text": entry["stage_text"],
            "section_key": entry["section_key"],
            "label": entry["label"],
            "event_date": entry["event_date"].isoformat(),
        },
        "item_attrs": transcript_payload["item_attrs"],
        "paragraph_count": transcript_payload["paragraph_count"],
        "char_count": transcript_payload["char_count"],
        "is_preliminary": transcript_payload["is_preliminary"],
        "transcript_text": transcript_payload["transcript_text"],
    }

    # json.dumps is used to convert the bundle dictionary into a JSON string
    json_path.write_text(
        json.dumps(bundle, ensure_ascii=False, indent=2), #ensure_ascii=False to allow non-ASCII characters to be written as they are (like accented characters in Spanish), indent = 2 because it makes the JSON file more readable 
        encoding="utf-8"
    )

    return txt_path, json_path

# having a summary csv help to have a file tracking which laws have been scraped already
def update_summary_csv(summary_csv_path, rows):
    """
    Create (or update) the global summary CSV, where information about all the discussions is stored. Deduplicate by history_url + section_key.
    """
    new_df = pd.DataFrame(rows)

    # if the summary CSV already exists, we load it and concatenate it with the new data
    if summary_csv_path.exists():
        old_df = pd.read_csv(summary_csv_path)
        full_df = pd.concat([old_df, new_df], ignore_index=True)
        full_df = full_df.drop_duplicates(subset=["history_url", "section_key"], keep="last") # so we don't duplicate discussions
    else:
        full_df = new_df.copy()

    full_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")
    return full_df

#### Full execution

We apply the whole pipeline for one law, this can be applied for several laws later

In [ ]:
def run_one_law(law_url):
    """Run the whole extraction pipeline for one law, the steps are:
    1. open a session
    2. download the page and get the metadata
    3. create the law folder
    4. discover all discussion entries
    5. filter to the target ones
    6. fetch each target transcript and save it
    7. return a list of summary rows to be added to the global summary CSV
    
    """
    print(f"Processing: {law_url}")

    # make one session per law to be more efficient and also to keep the cookies (if any) during the requests for the same law
    session = requests.Session()
    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept-Language": "es-CL,es;q=0.9,en;q=0.8",
    })


    # Download the page and get the metadata
    soup, raw_html, law_meta = get_law_page_metadata(session, law_url)
    
    #create the folder for the law
    law_folder = get_law_folder(BASE_DIR, law_meta)

    # print("Law number:", law_meta["law_number"])
    # print("Law title :", law_meta["law_title"])

    # this is optional, but helps for debugging
    if SAVE_RAW_HTML:
        (law_folder / "raw_page.html").write_text(raw_html, encoding="utf-8")

    # read the page index and discover all discussion entries
    all_discussions = discover_discussions_from_index(soup)

    # Filter to the target ones
    target_discussions = filter_target_discussions(all_discussions)
    # print(f"Found {len(all_discussions)} discussion entries, keeping {len(target_discussions)}")

    summary_rows = []

    # Convert the filtered DataFrame into a list of dicts so it is easier to loop through.
    target_records = target_discussions.to_dict(orient="records")

    # loop through the target records and fetch each transcript
    for i, entry in enumerate(target_records, start=1):
        # Build the file names for this discussion.
        stamp = entry["event_date"].isoformat()
        key = entry["section_key"].replace("-", "_")

        txt_path = law_folder / f"discussion_{stamp}_{key}.txt"
        json_path = law_folder / f"discussion_{stamp}_{key}.json"

        # Create the base summary row with the information we already have, so we can fill the rest of the fields later depending on whether we skip or fetch the transcript
        row = {
            "law_id": law_meta["law_id"],
            "law_number": law_meta["law_number"],
            "law_title": law_meta["law_title"],
            "history_url": law_meta["history_url"],
            "stage_num": entry["stage_num"],
            "stage_text": entry["stage_text"],
            "section_key": entry["section_key"],
            "event_date": entry["event_date"].isoformat(),
            "label": entry["label"],
            "parteDocumento": None,
            "modal_fecha": None,
            "paragraph_count": None,
            "char_count": None,
            "is_preliminary": None,
            "keep_for_analysis": None,
            "status": None,
            "txt_path": str(txt_path),
            "json_path": str(json_path),
        }


        # If we want to skip already saved files, read the existing JSON, and build the summary row from that saved information
        if SKIP_IF_ALREADY_SAVED and json_path.exists():
            saved = json.loads(json_path.read_text(encoding="utf-8"))

            # we update the base row with the information we get from the saved JSON
            row.update({
                "parteDocumento": saved.get("item_attrs", {}).get("partedocumento"),
                "modal_fecha": saved.get("item_attrs", {}).get("fecha"),
                "paragraph_count": saved.get("paragraph_count"),
                "char_count": saved.get("char_count"),
                "is_preliminary": saved.get("is_preliminary"),
                "keep_for_analysis": not bool(saved.get("is_preliminary")),
                "status": "already_saved",
            })

            summary_rows.append(row)
            continue


        # now we try to fecth and save the transcript. The point of the try if that if something fails during the fetching or saving, we can still have a summary row with the information we have
        try:
            transcript_payload = fetch_discussion_transcript(
                session=session,
                law_url=law_url,
                section_key=entry["section_key"],
            )

            # now we save the transcript and the metadata in txt and json files
            txt_path, json_path = save_discussion_file(
                law_folder=law_folder,
                law_meta=law_meta,
                entry=entry,
                transcript_payload=transcript_payload
            )

            # we update with the information we get from the transcript payload, which is the result of fetching the transcript
            row.update({
                "parteDocumento": transcript_payload["item_attrs"].get("partedocumento"),
                "modal_fecha": transcript_payload["item_attrs"].get("fecha"),
                "paragraph_count": transcript_payload["paragraph_count"],
                "char_count": transcript_payload["char_count"],
                "is_preliminary": transcript_payload["is_preliminary"],
                "keep_for_analysis": not transcript_payload["is_preliminary"],
                "status": "retrieved",
                "txt_path": str(txt_path),
                "json_path": str(json_path),
            })

        # with this except we catch any error that happens during the fetching or saving of the transcript
        except Exception as e:
            row.update({
                "status": f"failed: {e}",
                "txt_path": str(txt_path),
                "json_path": str(json_path),
            })
            print(f"Failed section {entry['section_key']}: {e}")

        summary_rows.append(row)

        # Pause between transcript POSTs
        if i < len(target_records):
            pause_random(PAUSE_BETWEEN_POSTS)

    return summary_rows

In [ ]:
# Apply on a single law
test_law = run_one_law("https://www.bcn.cl/historiadelaley/nc/historia-de-la-ley/8436/")

if test_law:
    summary_df = update_summary_csv(SUMMARY_CSV, test_law)
    display(summary_df.tail(len(test_law)))

In [ ]:
# run on multiple laws

for i, law_url in enumerate(LAW_URLS, start=1): # start=1 so there's no unnecessary waiting before i = len(LAW_URLS)
    rows = run_one_law(law_url)
    if rows:
        update_summary_csv(SUMMARY_CSV, rows)

    if i < len(LAW_URLS): # while we have more laws to process, we pause between them
        pause_random(PAUSE_BETWEEN_LAWS)

if SUMMARY_CSV.exists():
    summary_df = pd.read_csv(SUMMARY_CSV)
    display(summary_df)